In [2]:
# Notebook 03 - Feature Engineering (step-by-step)

# Cell A - Setup & Config
# - Import libs: pandas, numpy, ast, sklearn, scipy.sparse, pathlib, json, joblib
# - Define config: input/output paths, seed, TF-IDF params, topk_similarity

# Cell B - Load processed data
# - Load data/processed/recipes_clean.csv
# - Load data/processed/interactions_clean.csv
# - Parse list-like columns: ingredients_clean, tags_clean

# Cell C - Schema checks & hard guards
# - Validate dtypes: id, user_id, recipe_id, rating, minutes, calories
# - Remove orphan interactions (recipe_id not in recipes)
# - Enforce feature ranges (minutes > 0, calories in [10, 5000] or flag outlier)

# Cell D - Anti-leakage split
# - Split interactions: train/validation (time-based or leave-last-1 per user)
# - Use train_interactions to compute user/popularity statistics
# - Do not use validation interactions when fitting user-level stats

# Cell E - Text feature construction
# - Build ingredients_text, tags_text, description_clean
# - Build combined_text with deterministic weighting strategy

# Cell F - TF-IDF recipe vectors
# - Fit TfidfVectorizer on combined_text
# - Build recipe_tfidf_matrix + recipe_id/index mapping

# Cell G - Content similarity artifacts
# - Compute top-k cosine neighbors per recipe (avoid full NxN persistence)
# - Output columns: recipe_id, neighbor_recipe_id, similarity

# Cell H - Numerical & constraint features
# - Build rule-based fields: calories, minutes, n_ingredients_calc
# - Build buckets/flags: quick_meal, low_calorie, etc.

# Cell I - User features (from train only)
# - avg_rating_given, rating_count, activity_level
# - favorite_tags from high-rated history
# - default fallback for new users

# Cell J - Recipe popularity features
# - rating_count, rating_mean, popularity_score (Bayesian smoothing)
# - is_cold_item flag for cold-start logic

# Cell K - Feature validation checklist
# - Validate nulls, schema, key uniqueness
# - Coverage checks: users/features, recipes/vectors, recipes/top-k neighbors
# - Print final sanity summary

# Cell L - Save artifacts + manifest
# - Save into artifacts/features/
#   tfidf_vectorizer.joblib
#   recipe_tfidf_matrix.npz
#   recipe_index_map.parquet
#   recipe_similarity_topk.parquet
#   recipe_features.parquet
#   user_features.parquet
#   popularity_features.parquet
#   feature_manifest.json

# Output
# - Print artifact paths, shapes, coverage, and build timestamp

In [3]:
import pandas as pd
import numpy as np
import re
import ast
import json
import joblib
from pathlib import Path
from datetime import datetime
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [4]:
df_recipes = pd.read_csv("../data/processed/recipes_clean.csv")
df_interactions = pd.read_csv("../data/processed/interactions_clean.csv")

def parse_list_safe(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        try:
            v = ast.literal_eval(x.strip())
            return v if isinstance(v, list) else []
        except (ValueError, SyntaxError):
            return []
    return []

for col in ["ingredients_clean", "tags_clean", "ingredients", "tags", "steps"]:
    if col in df_recipes.columns:
        df_recipes[col] = df_recipes[col].apply(parse_list_safe)

print(f"recipes:      {df_recipes.shape}")
print(f"interactions: {df_interactions.shape}")
print(f"Sample ingredients_clean: {df_recipes['ingredients_clean'].iloc[0][:3]}")
print(f"Sample tags_clean:        {df_recipes['tags_clean'].iloc[0][:3]}")


recipes:      (77300, 24)
interactions: (706514, 5)
Sample ingredients_clean: ['winter squash', 'mexican seasoning', 'mixed spice']
Sample tags_clean:        ['60-minutes-or-less', 'time-to-make', 'course']


In [5]:
# === Cell C - Schema checks & hard guards ===

print("--- Before guards ---")
print(f"recipes:      {df_recipes.shape}")
print(f"interactions: {df_interactions.shape}")

# 1) Ép kiểu dữ liệu core
df_recipes["id"] = pd.to_numeric(df_recipes["id"], errors="coerce").astype("Int64")
df_recipes["minutes"] = pd.to_numeric(df_recipes["minutes"], errors="coerce")
df_recipes["calories"] = pd.to_numeric(df_recipes["calories"], errors="coerce")

df_interactions["user_id"] = pd.to_numeric(df_interactions["user_id"], errors="coerce").astype("Int64")
df_interactions["recipe_id"] = pd.to_numeric(df_interactions["recipe_id"], errors="coerce").astype("Int64")
df_interactions["rating"] = pd.to_numeric(df_interactions["rating"], errors="coerce")
if "date" in df_interactions.columns:
    df_interactions["date"] = pd.to_datetime(df_interactions["date"], errors="coerce")

# 2) Loại orphan interactions (recipe_id không có trong recipes)
valid_ids = set(df_recipes["id"].dropna())
n_before = len(df_interactions)
df_interactions = df_interactions[df_interactions["recipe_id"].isin(valid_ids)].copy()
print(f"Orphan interactions removed: {n_before - len(df_interactions):,}")

# 3) Filter range hợp lý cho feature
df_recipes = df_recipes[df_recipes["minutes"].between(1, 1440)].copy()
df_recipes["is_calorie_outlier"] = ~df_recipes["calories"].between(10, 5000)
n_outlier = df_recipes["is_calorie_outlier"].sum()
print(f"Calorie outliers flagged (kept but flagged): {n_outlier:,}")

# 4) Sync lại sau filter
df_interactions = df_interactions[df_interactions["recipe_id"].isin(set(df_recipes["id"]))].copy()

print("\n--- After guards ---")
print(f"recipes:      {df_recipes.shape}")
print(f"interactions: {df_interactions.shape}")
print(f"unique users: {df_interactions['user_id'].nunique():,}")
print(f"unique items: {df_interactions['recipe_id'].nunique():,}")
print(f"orphan check: {len(set(df_interactions['recipe_id']) - set(df_recipes['id']))} orphans")

--- Before guards ---
recipes:      (77300, 24)
interactions: (706514, 5)
Orphan interactions removed: 7,138
Calorie outliers flagged (kept but flagged): 692

--- After guards ---
recipes:      (77300, 25)
interactions: (699376, 5)
unique users: 30,836
unique items: 77,300
orphan check: 0 orphans


In [6]:
# === Cell D - Anti-leakage split ===
# Split interactions theo thời gian: 80% cũ nhất -> train, 20% mới nhất -> validation
# Chỉ dùng train_interactions để tính user/popularity stats phía sau

interactions_sorted = df_interactions.sort_values("date").reset_index(drop=True)
split_idx = int(len(interactions_sorted) * 0.8)

train_interactions = interactions_sorted.iloc[:split_idx].copy()
val_interactions = interactions_sorted.iloc[split_idx:].copy()

train_user_ids = set(train_interactions["user_id"])
train_recipe_ids = set(train_interactions["recipe_id"])
val_user_ids = set(val_interactions["user_id"])
val_recipe_ids = set(val_interactions["recipe_id"])

cold_start_users = val_user_ids - train_user_ids
cold_start_items = val_recipe_ids - train_recipe_ids

print(f"train_interactions: {len(train_interactions):,} rows")
print(f"val_interactions:   {len(val_interactions):,} rows")
print(f"train date range:   {train_interactions['date'].min()} -> {train_interactions['date'].max()}")
print(f"val   date range:   {val_interactions['date'].min()} -> {val_interactions['date'].max()}")
print(f"\ntrain users: {len(train_user_ids):,}  |  train items: {len(train_recipe_ids):,}")
print(f"val   users: {len(val_user_ids):,}  |  val   items: {len(val_recipe_ids):,}")
print(f"cold-start users in val: {len(cold_start_users):,}")
print(f"cold-start items in val: {len(cold_start_items):,}")

train_interactions: 559,500 rows
val_interactions:   139,876 rows
train date range:   2000-02-25 00:00:00 -> 2010-08-14 00:00:00
val   date range:   2010-08-14 00:00:00 -> 2018-12-19 00:00:00

train users: 27,021  |  train items: 72,860
val   users: 14,375  |  val   items: 44,122
cold-start users in val: 3,815
cold-start items in val: 4,440


In [7]:
# === Cell E ===
# plan Cell E : Xác định các nguồn text trong dataset --> NormalizeText --> Convert list --> Tạo text
def normalize_text(text):
    """
    Normalize general text fields (description)
    """
    if pd.isna(text):
        return ""
    
    text = text.lower()
    
    # remove punctuation
    text = re.sub(r"[^\w\s]", " ", text)
    
    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text


def normalize_ingredient(token):
    """
    Normalize ingredient tokens
    """
    if pd.isna(token):
        return ""
    
    token = token.lower().strip()
    
    # replace spaces with underscore
    token = token.replace(" ", "_")
    
    return token


def list_to_text(lst):
    """
    Convert list column to normalized text
    """
    if not isinstance(lst, list):
        return ""
    
    tokens = [normalize_ingredient(x) for x in lst]
    
    return " ".join(tokens)


def tags_to_text(lst):
    """
    Convert tags list to normalized text
    """
    if not isinstance(lst, list):
        return ""
    
    tokens = [normalize_text(x) for x in lst]
    
    return " ".join(tokens)


# ---------- Build text features ----------

# ingredients text
df_recipes["ingredients_text"] = df_recipes["ingredients_clean"].apply(list_to_text)

# tags text
df_recipes["tags_text"] = df_recipes["tags_clean"].apply(tags_to_text)

# description text
if "description" in df_recipes.columns:
    df_recipes["description_clean"] = df_recipes["description"].apply(normalize_text)
else:
    df_recipes["description_clean"] = ""


# ---------- Build combined_text with weighting ----------

def build_combined_text(row):
    
    ingredients = row["ingredients_text"]
    tags = row["tags_text"]
    description = row["description_clean"]
    
    combined = " ".join([
        ingredients,        # ingredients weight 1
        ingredients,        # ingredients weight 2
        tags,               # tags weight 1
        tags,               # tags weight 2
        description         # description weight 1
    ])
    
    return combined


df_recipes["combined_text"] = df_recipes.apply(build_combined_text, axis=1)


# ---------- Basic checks ----------

print("Text feature construction completed\n")

print("Example rows:\n")
display(
    df_recipes[
        ["id","ingredients_text","tags_text","description_clean","combined_text"]
    ].head()
)

print("\nNull counts:")
print(df_recipes[["ingredients_text","tags_text","combined_text"]].isnull().sum())

print("\nAverage text length:")
print(df_recipes["combined_text"].str.len().mean())

Text feature construction completed

Example rows:



,id,ingredients_text,tags_text,description_clean,combined_text
0,137739,winter_squash mexican_seasoning mixed_spice ho...,60 minutes or less time to make course main in...,autumn is my favorite time of year to cook thi...,winter_squash mexican_seasoning mixed_spice ho...
1,75452,sugar unsalted_butter bananas eggs fresh_lemon...,weeknight time to make course main ingredient ...,from ann hodgman s,sugar unsalted_butter bananas eggs fresh_lemon...
2,63986,lean_pork_chops flour salt dry_mustard garlic_...,weeknight time to make course main ingredient ...,here s and old standby i enjoy from time to ti...,lean_pork_chops flour salt dry_mustard garlic_...
3,43026,egg_roll_wrap whole_green_chilies cheese corns...,60 minutes or less time to make course main in...,a favorite from a local restaurant no longer i...,egg_roll_wrap whole_green_chilies cheese corns...
4,23933,butterscotch_chips chinese_noodles salted_peanuts,15 minutes or less time to make course prepara...,a little different and oh so good i include th...,butterscotch_chips chinese_noodles salted_pean...



Null counts:
ingredients_text    0
tags_text           0
combined_text       0
dtype: int64

Average text length:
844.5295601552393


In [8]:
print(df_recipes["combined_text"].isnull().sum())
print(df_recipes["combined_text"].sample(5))
print(df_recipes["combined_text"].str.split().explode().value_counts().head(10))

0
64195    flour toasted_wheat_germ cinnamon salt baking_...
6149     flour canola_oil celery bell_pepper yellow_oni...
62597    crabmeat green_onion celery low-fat_mayonnaise...
32191    zucchini onions sweet_red_peppers salt cider_v...
31674    cornmeal unbleached_flour baking_powder salt s...
Name: combined_text, dtype: str


combined_text
low            248594
or             239368
to             230029
less           201544
make           172018
main           169126
time           165680
preparation    155027
course         147578
and            121087
Name: count, dtype: int64


In [9]:
# === Cell F - TF-IDF Recipe Vectors ===
#
# Mục tiêu: biến combined_text của mỗi recipe thành vector số để tính similarity.
#
# Pipeline:
#   combined_text (77k strings)
#       -> TfidfVectorizer.fit_transform()
#           -> tfidf_vectorizer : model đã học vocabulary (token -> column index)
#           -> recipe_tfidf_matrix : sparse matrix (n_recipes x n_features), mỗi hàng = 1 recipe vector
#       -> recipe_index_map : bảng ánh xạ recipe_id <-> row index trong matrix
#
# Tham số TF-IDF:
#   max_features=20000  : giới hạn vocab size, giữ top 20k token theo tần suất
#   ngram_range=(1,2)   : bắt cả unigram (chicken) lẫn bigram (chicken_breast)
#   min_df=5            : bỏ token xuất hiện < 5 recipes (quá hiếm, không khái quát)
#   max_df=0.95         : bỏ token xuất hiện > 95% recipes (stopword-like: "low", "to", "or")
#   sublinear_tf=True   : dùng 1+log(tf) thay raw count, giảm bias cho text dài

tfidf_vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.95,
    sublinear_tf=True,
)

recipe_tfidf_matrix = tfidf_vectorizer.fit_transform(df_recipes["combined_text"])

# Ánh xạ recipe_id <-> vị trí hàng trong matrix, dùng để tra cứu ở Cell G và notebook 04
recipe_index_map = pd.DataFrame({
    "recipe_id": df_recipes["id"].values,
    "matrix_index": range(len(df_recipes))
})

# ---------- Sanity check ----------
print(f"TF-IDF matrix shape: {recipe_tfidf_matrix.shape}")
print(f"Vocabulary size:     {len(tfidf_vectorizer.vocabulary_)}")
print(f"Sparsity:            {1 - recipe_tfidf_matrix.nnz / (recipe_tfidf_matrix.shape[0] * recipe_tfidf_matrix.shape[1]):.4%}")

vocab = tfidf_vectorizer.get_feature_names_out()
print(f"\nSample vocab (first 10):  {list(vocab[:10])}")
print(f"Sample vocab (last 10):   {list(vocab[-10:])}")
print(f"\nrecipe_index_map:")
display(recipe_index_map.head())


TF-IDF matrix shape: (77300, 20000)
Vocabulary size:     20000
Sparsity:            99.4729%

Sample vocab (first 10):  ['05', '06', '07', '08', '09', '10', '10 minutes', '10 years', '100', '11']
Sample vocab (last 10):   ['zucchini red_bell_pepper', 'zucchini salt', 'zucchini tomatoes', 'zucchini yellow_squash', 'zwt', 'zwt region', 'zwt3', 'zwt5', 'zwt6', 'zwt7']

recipe_index_map:


,recipe_id,matrix_index
0,137739,0
1,75452,1
2,63986,2
3,43026,3
4,23933,4


In [10]:
# Cell G - Content similarity artifacts
# - Compute top-k cosine neighbors per recipe (avoid full NxN persistence)
# - Output columns: recipe_id, neighbor_recipe_id, similarity
TOP_K = 20
BATCH_SIZE = 500
id_array = recipe_index_map["recipe_id"].values
rows = []
n_recipes = recipe_tfidf_matrix.shape[0]
for start in range(0, n_recipes, BATCH_SIZE):
    end = min(start + BATCH_SIZE, n_recipes)
    # Cosine similarity của batch này vs toàn bộ recipes
    sim_batch = cosine_similarity(recipe_tfidf_matrix[start:end], recipe_tfidf_matrix)
    for i, sim_row in enumerate(sim_batch):
        idx = start + i
        # Lấy top-k+1 (bỏ chính nó), argsort descending
        top_indices = np.argsort(sim_row)[::-1][1:TOP_K + 1]
        for neighbor_idx in top_indices:
            rows.append({
                "recipe_id": id_array[idx],
                "neighbor_recipe_id": id_array[neighbor_idx],
                "similarity": sim_row[neighbor_idx]
            })
    if (start // BATCH_SIZE) % 20 == 0:
        print(f"  processed {end:,}/{n_recipes:,} recipes...")
recipe_similarity_topk = pd.DataFrame(rows)
print(f"\nrecipe_similarity_topk shape: {recipe_similarity_topk.shape}")
print(f"Expected rows: {n_recipes * TOP_K:,}")
print(f"\nSample:")
display(recipe_similarity_topk.head(10))
print(f"\nSimilarity stats:")
print(recipe_similarity_topk["similarity"].describe())


  processed 500/77,300 recipes...
  processed 10,500/77,300 recipes...
  processed 20,500/77,300 recipes...
  processed 30,500/77,300 recipes...
  processed 40,500/77,300 recipes...
  processed 50,500/77,300 recipes...
  processed 60,500/77,300 recipes...
  processed 70,500/77,300 recipes...

recipe_similarity_topk shape: (1546000, 3)
Expected rows: 1,546,000

Sample:


,recipe_id,neighbor_recipe_id,similarity
0,137739,235285,0.289120
1,137739,216993,0.286754
2,137739,189352,0.282295
3,137739,267785,0.277842
4,137739,30377,0.262536
5,137739,96811,0.262359
6,137739,102104,0.256990
7,137739,143212,0.256736
8,137739,110726,0.253018
9,137739,82777,0.252915



Similarity stats:
count    1.546000e+06
mean     3.047163e-01
std      7.680457e-02
min      1.036201e-01
25%      2.520061e-01
50%      2.953698e-01
75%      3.464550e-01
max      9.919496e-01
Name: similarity, dtype: float64


In [13]:
# Cell H - Numerical & constraint features
# - Build rule-based fields: calories, minutes, n_ingredients_calc
# - Build buckets/flags: quick_meal, low_calorie, etc.
# Thời gian nấu
df_recipes["is_quick_meal"] = df_recipes["minutes"] <= 30
df_recipes["is_medium_meal"] = df_recipes["minutes"].between(31, 60)
df_recipes["is_long_meal"] = df_recipes["minutes"] > 60

# Calories
df_recipes["is_low_calorie"] = df_recipes["calories"].between(10, 300)
df_recipes["is_medium_calorie"] = df_recipes["calories"].between(301, 600)
df_recipes["is_high_calorie"] = df_recipes["calories"] > 600

# Độ phức tạp theo số nguyên liệu
df_recipes["is_simple_recipe"] = df_recipes["n_ingredients_calc"] <= 5
df_recipes["is_complex_recipe"] = df_recipes["n_ingredients_calc"] >= 15

recipe_features = df_recipes[[
    "id", "minutes", "calories", "n_ingredients_calc",
    "is_calorie_outlier",
    "is_quick_meal", "is_medium_meal", "is_long_meal",
    "is_low_calorie", "is_medium_calorie", "is_high_calorie",
    "is_simple_recipe", "is_complex_recipe"
]].copy()
recipe_features = recipe_features.rename(columns={"id": "recipe_id"})

print(f"recipe_features shape: {recipe_features.shape}")
print(f"\nDistribution:")
print(f"  quick_meal:    {recipe_features['is_quick_meal'].sum():,} ({recipe_features['is_quick_meal'].mean():.1%})")
print(f"  low_calorie:   {recipe_features['is_low_calorie'].sum():,} ({recipe_features['is_low_calorie'].mean():.1%})")
print(f"  simple_recipe: {recipe_features['is_simple_recipe'].sum():,} ({recipe_features['is_simple_recipe'].mean():.1%})")
print(f"\nSample:")
display(recipe_features.head())

recipe_features shape: (77300, 13)

Distribution:
  quick_meal:    34,877 (45.1%)
  low_calorie:   38,104 (49.3%)
  simple_recipe: 14,251 (18.4%)

Sample:


,recipe_id,minutes,calories,n_ingredients_calc,is_calorie_outlier,is_quick_meal,is_medium_meal,is_long_meal,is_low_calorie,is_medium_calorie,is_high_calorie,is_simple_recipe,is_complex_recipe
0,137739,55,51.5,7,False,False,True,False,True,False,False,False,False
1,75452,70,2669.3,9,False,False,False,True,False,False,True,False,False
2,63986,500,105.7,7,False,False,False,True,True,False,False,False,False
3,43026,45,94.0,5,False,False,True,False,True,False,False,True,False
4,23933,15,232.7,3,False,True,False,False,True,False,False,True,False


In [14]:
# Cell I - User features (from train only)
# - avg_rating_given, rating_count, activity_level
# - favorite_tags from high-rated history
# - default fallback for new users
# 1) Thống kê cơ bản theo user
user_stats = train_interactions.groupby("user_id").agg(
    avg_rating_given=("rating", "mean"),
    rating_count=("rating", "count"),
    min_date=("date", "min"),
    max_date=("date", "max"),
).reset_index()
# Activity level: phân tier theo số lượng ratings
user_stats["activity_level"] = pd.cut(
    user_stats["rating_count"],
    bins=[0, 5, 20, np.inf],
    labels=["cold", "warm", "active"]
)
# 2) Favorite tags: top-3 tags phổ biến nhất từ recipes user đã rating >= 4
high_rated = train_interactions[train_interactions["rating"] >= 4][["user_id", "recipe_id"]]
high_rated = high_rated.merge(
    df_recipes[["id", "tags_clean"]].rename(columns={"id": "recipe_id"}),
    on="recipe_id", how="left"
)
def get_top_tags(tags_series, top_n=3):
    from collections import Counter
    all_tags = []
    for tag_list in tags_series:
        if isinstance(tag_list, list):
            all_tags.extend(tag_list)
    if not all_tags:
        return []
    return [t for t, _ in Counter(all_tags).most_common(top_n)]
favorite_tags = high_rated.groupby("user_id")["tags_clean"].apply(get_top_tags).reset_index()
favorite_tags.columns = ["user_id", "favorite_tags"]
# 3) Merge thành user_features
user_features = user_stats.merge(favorite_tags, on="user_id", how="left")
user_features["favorite_tags"] = user_features["favorite_tags"].apply(
    lambda x: x if isinstance(x, list) else []
)
# 4) Global fallback cho user mới (không có trong train)
global_avg_rating = train_interactions["rating"].mean()
global_top_tags = get_top_tags(df_recipes["tags_clean"], top_n=3)
print(f"Global fallback: avg_rating={global_avg_rating:.2f}, top_tags={global_top_tags}")
print(f"\nuser_features shape: {user_features.shape}")
print(f"\nActivity level distribution:")
print(user_features["activity_level"].value_counts())
print(f"\nSample:")
display(user_features.head())

Global fallback: avg_rating=4.71, top_tags=['preparation', 'time-to-make', 'course']

user_features shape: (27021, 7)

Activity level distribution:
activity_level
cold      13500
warm       9021
active     4500
Name: count, dtype: int64

Sample:


,user_id,avg_rating_given,rating_count,min_date,max_date,activity_level,favorite_tags
0,1533,4.849315,73,2002-02-19,2008-03-01,active,"[time-to-make, preparation, course]"
1,1535,4.563771,541,2004-05-22,2010-08-14,active,"[preparation, time-to-make, course]"
2,1634,4.342857,35,2001-07-05,2010-02-05,active,"[time-to-make, preparation, dietary]"
3,1676,4.500000,16,2002-07-24,2010-04-23,warm,"[main-ingredient, preparation, time-to-make]"
4,1773,4.000000,3,2000-03-13,2001-01-29,cold,"[weeknight, time-to-make, course]"
